# Paper Experiment — Artifact Report

Notebook này **không huấn luyện mô hình** và không chứa implementation thứ hai. Nguồn authoritative duy nhất:

```bash
python scripts/run_paper_experiment.py --config configs/experiment.yaml
```

Chuỗi phương pháp khóa: H → B1 → B2 → M1 → M2. Gold graded 0–3 chỉ dùng final evaluation; MAP dùng `relevance >= 2`; CI chính paired bootstrap trên graded `job_id`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'

required = [
    'tables/main_results.csv',
    'tables/bootstrap_results.csv',
    'tables/perturbation_results.csv',
    'audit/run_manifest.json',
    'audit/preprocessing_manifest.json',
    'audit/pair_hashes.csv',
]
missing = [name for name in required if not (RESULTS / name).exists()]
assert not missing, f'Missing authoritative artifacts: {missing}'

run_manifest = json.loads((RESULTS / 'audit/run_manifest.json').read_text(encoding='utf-8'))
preprocessing = json.loads((RESULTS / 'audit/preprocessing_manifest.json').read_text(encoding='utf-8'))
pair_hashes = pd.read_csv(RESULTS / 'audit/pair_hashes.csv')
main_results = pd.read_csv(RESULTS / 'tables/main_results.csv')
bootstrap_results = pd.read_csv(RESULTS / 'tables/bootstrap_results.csv')
perturbation_results = pd.read_csv(RESULTS / 'tables/perturbation_results.csv')

assert pair_hashes['m1_train_pair_hash'].equals(pair_hashes['m2_train_pair_hash'])
assert pair_hashes['m1_validation_pair_hash'].equals(pair_hashes['m2_validation_pair_hash'])
assert run_manifest['counts']['sampled_pairs'] == 16000
assert run_manifest['counts']['graded_pairs'] == 100
assert run_manifest['counts']['graded_jobs'] == 12
print('run_mode:', run_manifest['run_mode'])
if run_manifest['run_mode'] != 'full':
    print('WARNING: artifacts are smoke-only and must not be used as paper numbers.')
display(pd.DataFrame([run_manifest['counts']]))

## RQ1 — Ranking trên graded evaluation set

Bảng chính chỉ gồm H/B1/B2/M1. Không diễn giải một hiệu số là cải thiện chắc chắn khi paired-JD 95% CI chứa 0.

In [ ]:
display(main_results)
display(bootstrap_results)

rq1_interpretation = bootstrap_results.copy()
rq1_interpretation['ci_excludes_zero'] = (
    (rq1_interpretation['ci_95_low'] > 0) |
    (rq1_interpretation['ci_95_high'] < 0)
)
rq1_interpretation['interpretation'] = np.where(
    rq1_interpretation['ci_excludes_zero'],
    'CI excludes 0',
    'CI contains 0 — improvement not established',
)
display(rq1_interpretation)

## RQ2 — Isolated qualification perturbations

QualSens được báo riêng cho skill, experience và domain. M2 chỉ khác M1 bởi auxiliary skill-gap loss; CI M2−M1 được bootstrap theo graded JD.

In [ ]:
display(perturbation_results)
rq2 = perturbation_results.copy()
rq2['ci_excludes_zero'] = (rq2['ci_95_low'] > 0) | (rq2['ci_95_high'] < 0)
display(rq2)

## Reproducibility and leakage audit

- TF–IDF vocabulary/IDF và optional skill-frequency filter fit trên TRAIN only.
- Graded JD và graded CV không thuộc weak-label pool hoặc preprocessing fit scope.
- Validation/test/gold chỉ transform bằng frozen preprocessing/LF/Dawid–Skene state.
- M1/M2 dùng cùng train và validation pair-table hashes.
- Kết quả smoke không phải paper numbers.

In [ ]:
display(pair_hashes)
display(pd.DataFrame([{
    'fit_jobs': len(preprocessing['fit_job_ids']),
    'fit_candidates': len(preprocessing['fit_candidate_ids']),
    'role_vocabulary_size': preprocessing['role_vocabulary_size'],
    'description_vocabulary_size': preprocessing['desc_vocabulary_size'],
    'high_df_skill_count': len(preprocessing['high_df_skills']),
}]))
print('source hashes')
display(pd.Series(run_manifest['source_files'], name='sha256').to_frame())

## Interpretation limits

1. Không có hiring outcome; `y_prob` chỉ là weak relevance signal.
2. LF phụ thuộc cùng năm feature đầu vào.
3. B1 học từ nhãn nhị phân suy ra từ H và chỉ là ablation.
4. Graded set có 100 cặp, 12 JD, một annotator; statistical power thấp.
5. RankNet MLP là relevance function, không phải bộ trọng số tối ưu.
6. Null hoặc negative result phải được giữ nguyên.